In [ ]:
#%pip install pipreqs

#!python -m venv venv
#!venv\Scripts\python.exe -m pip install pipreqs
#!venv/Scripts/pipreqs.exe .

# Descrição do Projeto

A empresa Sweet Lift Taxi coletou dados históricos sobre pedidos de táxi nos aeroportos. Para atrair mais motoristas durante o horário de pico, precisamos prever a quantidade de pedidos de táxi para a próxima hora. 	
Construa um modelo para tal predição.

A métrica REQM no conjunto de teste não deve ser superior a 48.

## Instruções do projeto

1. Faça download dos dados e faça uma nova amostragem em uma hora.
2. Analise os dados
3. Treine diferentes modelos com diferentes hiperparâmetros. A amostra de teste deve ser 10% do conjunto de dados inicial.
4. Teste os dados usando a amostra de teste e forneça uma conclusão.

## Descrição dos dados

Os dados são armazenados no arquivo `taxi.csv`. O número de pedidos está na coluna `num_orders`.

## Preparação 

In [ ]:

%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error # Utilizando o legacy
from sklearn.metrics import mean_absolute_percentage_error # Para ver a percentagem de erro
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

# Ensure LightGBM is installed before the model is used
try:
    from lightgbm import LGBMRegressor
except ModuleNotFoundError:
    !python -m pip install lightgbm
    from lightgbm import LGBMRegressor
from lightgbm import LGBMRegressor

#######################################################################
df = pd.read_csv('data\\taxi.csv') # Load do dataset 
print(df.sample(5)) # Visualização do dataset de maneira aleatória
print("="*60)
#######################################################################
df['datetime'] = pd.to_datetime(df['datetime'])
df.set_index('datetime', inplace=True)
df_hourly=df.resample('1h').sum()
df_hourly.to_csv('data_hours.csv')
print('Nova amostragem concluída com sucesso')

## Análise

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

print(df_hourly.shape)
print("="*60)
print(df_hourly.info())
print("="*60)
print(df_hourly.describe().T)
########################################################

# 1. Executar a decomposição sazonal (ciclo de 24 horas)
decomposicao = seasonal_decompose(df_hourly['num_orders'], model='additive', period=24)

# ============================================================
# GRÁFICO 1: Série Original vs. Tendência (Substituindo a média móvel)
# ============================================================
fig, ax = plt.subplots(figsize=(15, 5))

decomposicao.observed.plot(ax=ax, alpha=0.4, label='Pedidos por hora (Observado)', color='steelblue')
decomposicao.trend.plot(ax=ax, label='Tendência (Seasonal Decompose)', color='red', linewidth=2)

ax.set_title('Pedidos de Táxi: Série Observada vs. Tendência')
ax.set_xlabel('Data')
ax.set_ylabel('Número de Pedidos')
ax.legend()
plt.tight_layout()
plt.show()

print("="*60)

# ============================================================
# GRÁFICO 2: Componentes Sazonal e Resíduo ao Longo do Tempo
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)

# Sazonalidade (mostra o padrão cíclico extraído)
# Dica: Se a série for muito longa, você pode fatiar (ex: decomposicao.seasonal['2018-08-01':'2018-08-07'])
decomposicao.seasonal.plot(ax=axes[0], color='green', linewidth=1)
axes[0].set_title('Componente Sazonal (Ciclo de 24h)')
axes[0].set_ylabel('Efeito Sazonal')

# Resíduos (ruído não explicado pela tendência nem pela sazonalidade)
decomposicao.resid.plot(ax=axes[1], color='purple', alpha=0.6, style='.')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Resíduos (Ruído Aleatório)')
axes[1].set_xlabel('Data')
axes[1].set_ylabel('Resíduo')

plt.tight_layout()
plt.show()

print("="*60)

# ============================================================
# GRÁFICO 3: Perfil Sazonal por Hora e Dia da Semana (Seus gráficos de barras)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Efeito sazonal médio por hora do dia (mostra quais horas puxam os pedidos para cima ou para baixo)
decomposicao.seasonal.groupby(decomposicao.seasonal.index.hour).mean().plot(
    kind='bar', ax=axes[0], color='steelblue'
)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Efeito Sazonal por Hora do Dia')
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('Variação em relação à Tendência')

# Média de pedidos por dia da semana (mantido do seu código original)
df_hourly.groupby(df_hourly.index.dayofweek)['num_orders'].mean().plot(
    kind='bar', ax=axes[1], color='coral'
)
axes[1].set_title('Média de Pedidos por Dia da Semana')
axes[1].set_xlabel('Dia (0=Segunda, 6=Domingo)')
axes[1].set_ylabel('Média de Pedidos')

plt.tight_layout()
plt.show()


## Treinamento

In [ ]:
df_model = df_hourly.copy()

# Features de lag
for lag in [1, 2, 3, 6, 12, 24]:
    df_model[f'lag_{lag}'] = df_model['num_orders'].shift(lag)

# Média móvel
df_model['rolling_mean_24'] = df_model['num_orders'].shift(1).rolling(24).mean()

# Additional features
df_model['day_of_week'] = df_model.index.dayofweek
df_model['hour'] = df_model.index.hour

# Remover linhas com NaN
df_model = df_model.dropna()

print(df_model.shape)
print(df_model.head())

# Dividir os dados em treino e teste (90% /  10% e deve ser cronológica)
## Devemos usar o passado para prever o futuro

split_index = int(len(df_model) * 0.9) # Dividir de maneira cronológica usando o index

train = df_model.iloc[:split_index]
test = df_model.iloc[split_index:]

# imprimindo os tamanhos 
print("="*60)
print(f'Treino: {train.shape}')
print("="*60)
print(f'Test: {test.shape}')


## Testando

In [ ]:
X_train = train.drop(columns='num_orders') # todas as colunas exceptuando o target
y_train = train['num_orders']

X_test = test.drop(columns='num_orders')
y_test = test['num_orders']

# Cálculo de Baseline (Referência para comparar os modelos)

pred_baseline = np.full(len(y_test), y_train.median())
rmse_baseline = np.sqrt(mean_squared_error(y_test, pred_baseline))
print(f"RMSE da Baseline: {rmse_baseline:.2f}")

# Treinar os vários modelos
## Primeiro modelo é o Linear Regression

### parametro 1: fit_intercept

configs_lr = [
    {"fit_intercept": True, "positive": False},
    {"fit_intercept": True, "positive": True}
]
rmse_lr = float('inf')  # Inicializa com infinito para buscar o menor valor
best_predict_lr=None
best_model_lr=None

for config in configs_lr:
    model_lr = LinearRegression(**config)
    model_lr.fit(X_train, y_train)
    predict_lr = model_lr.predict(X_test)
    rmse_atual = np.sqrt(mean_squared_error(y_test, predict_lr))
    
    # Guarda o menor RMSE encontrado entre as variações da regressão
    if rmse_atual < rmse_lr:
        rmse_lr = rmse_atual
        best_predict_lr=predict_lr
        best_model_lr=model_lr

## Segundo modelo RandomForestRegressor

configs_rfr = [
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 200, "max_depth": 15}
]
rmse_rfr = float('inf')
best_predict_rfr = None  # Guardando as previsões
best_model_rfr = None    # Guardando o objeto do modelo

for config in configs_rfr:
    model_rfr = RandomForestRegressor(**config, random_state=42, n_jobs=-1)
    model_rfr.fit(X_train, y_train)
    predict_rfr = model_rfr.predict(X_test)
    rmse_atual = np.sqrt(mean_squared_error(y_test, predict_rfr))
    
    # Atualiza se a configuração atual for melhor que a anterior
    if rmse_atual < rmse_rfr:
        rmse_rfr = rmse_atual            # Guardando o valor do erro
        best_predict_rfr = predict_rfr   # Guardando o array de previsões vencedor
        best_model_rfr = model_rfr       # Guardando o modelo treinado em si
## Terceiro modelo GradientBoostingRegressor

configs_gb = [
    {"n_estimators": 100, "learning_rate": 0.05},
    {"n_estimators": 150, "learning_rate": 0.1}
]
rmse_gb = float('inf')
best_predict_gb=None
best_model_gb=None

for config in configs_gb:
    model_gb = GradientBoostingRegressor(**config, random_state=42)
    model_gb.fit(X_train, y_train)
    predict_gb = model_gb.predict(X_test)
    rmse_atual = np.sqrt(mean_squared_error(y_test, predict_gb))
    
    if rmse_atual < rmse_gb:
        rmse_gb = rmse_atual
        best_predict_gb=predict_gb
        best_model_gb=model_gb

## Quarto modelo LightGBMRegressor

configs_lgbm = [
    {"n_estimators": 100, "learning_rate": 0.05, "num_leaves": 31},
    {"n_estimators": 200, "learning_rate": 0.1, "num_leaves": 50}
]
rmse_lgbm = float('inf')
best_predict_lgbm=None
best_model_lgbm=None

for config in configs_lgbm:
    # verbose=-1 apenas silencia avisos repetitivos do LightGBM no loop
    model_lgbm = LGBMRegressor(**config, random_state=42, verbose=-1)
    model_lgbm.fit(X_train, y_train)
    predict_lgbm = model_lgbm.predict(X_test)
    rmse_atual = np.sqrt(mean_squared_error(y_test, predict_lgbm))
    
    if rmse_atual < rmse_lgbm:
        rmse_lgbm = rmse_atual
        best_predict_lgbm=predict_lgbm
        best_model_lgbm=model_lgbm

# Resultados dos vários modelos de maneira dinámica com o melhor de cada:
## Dicionário com os valores das melhores predições
predictions_dict = {
    'LR': best_predict_lr,
    'RFR': best_predict_rfr,
    'GB': best_predict_gb,
    'LGBM': best_predict_lgbm
}

## passo 1:

results = {
    'LR': rmse_lr,
    'RFR': rmse_rfr,
    'GB': rmse_gb,
    'LGBM': rmse_lgbm,
    'Baseline': rmse_baseline
}

# Mantém as previsões alinhadas com os mesmos nomes do dicionário de resultados
predictions_dict['Baseline'] = pred_baseline

## passo 2: 
limite_rmse = 48

# Variáveis para capturar o campeão absoluto da rodada
best_rmse = float('inf')
best_model = None

print("--- Avaliação de Desempenho dos Modelos ---")
for model, valor in results.items():
    if valor <= limite_rmse:
        print(f'{model}: Dentro dos parâmetros (RMSE: {valor:.2f})')
    else: 
        print(f'{model}: Fora do parâmetro esperado (RMSE: {valor:.2f})')
        
    # Identifica dinamicamente quem obteve o menor erro do projeto
    if valor < best_rmse:
        best_rmse = valor
        best_model = model
        best_predictions = predictions_dict[model] # Extraindo o valor correct de maniera dinâmica

print("\n--- Conclusão da Busca ---")
print(f"🏆 O melhor modelo configurado foi o {best_model} com RMSE de {best_rmse:.2f}!")


In [ ]:
# Gráfico comparativo dos modelos

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

models_names = list(results.keys())
models_rmse = list(results.values())

# Identifica automaticamente o menor RMSE (o modelo vencedor)
menor_rmse = min(models_rmse)
indice_vencedor = models_rmse.index(menor_rmse)

for i, ax in enumerate(axes):
    # Lógica de cores: destaca o vencedor com Verde e os outros ficam em Cinza suave
    if i == indice_vencedor:
        cor_barra = '#2ecc71'  # Verde esmeralda para o vencedor
        label_barra = f'Vencedor ({models_names[i]})'
        # Adiciona uma borda na barra do vencedor para maior destaque
        borda = {'edgecolor': '#27ae60', 'linewidth': 2}
    else:
        cor_barra = '#b2bec3'  # Cinza neutro para os demais
        label_barra = 'Outros Modelos'
        borda = {}

    # Plota a barra individual
    barra = ax.bar(models_names[i], models_rmse[i], color=cor_barra, **borda, label=label_barra)
    
    # Linha do limite máximo aceitável (48)
    linha_limite = ax.axhline(y=48, color='#e74c3c', linestyle='--', linewidth=1.5, label='Limite Máximo (48)')
    
    # Mostra o valor numérico exato acima da barra
    ax.text(0, models_rmse[i] + 0.4, f'{models_rmse[i]:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # Títulos e formatação dos eixos
    ax.set_title(f'Modelo: {models_names[i]}', fontsize=12, fontweight='bold', pad=10)
    ax.set_ylabel('RMSE', fontsize=10)
    ax.set_ylim(35, 58)  # Ajustado o limite inferior para dar melhor proporção visual às barras
    
    # Remove marcações desnecessárias no eixo X já que o título identifica o modelo
    ax.set_xticks([]) 
    
    # Remove bordas superiores e direitas do gráfico (visual mais limpo)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Posiciona a legenda de forma organizada dentro de cada quadrante
    ax.legend(loc='upper right', fontsize=9, framealpha=0.8)

# Título principal com o anúncio do melhor modelo
modelo_vencedor_nome = models_names[indice_vencedor]
plt.suptitle(f'Comparação de RMSE por Modelo\n Melhor Escolha: {modelo_vencedor_nome} (RMSE: {menor_rmse:.2f})', 
             fontsize=15, fontweight='bold', color='#2c3e50', y=0.98)

plt.tight_layout()
plt.show()


In [ ]:
mape_decimal = mean_absolute_percentage_error(y_test, best_predictions)# Cálculo do mape decimal
mape_percentual = mape_decimal * 100 # Cálculo do mape percentual 

print(f"MAPE: {mape_percentual:.2f}%")

## Conclusão Geral

- Este projecto tinha como principal objectivo, prever a quantidade de pedidos de táxi para a próxima hora da empresa de "Sweet Lift Taxi", analisando os dados históricos das várias corridas. Para tal foi necessário fazer uma reamostragem utilizando o resample para que cada ponto dos dados originais fique dentro de intervalos de uma hora;  

- Foram escolhidos 4 modelos para este projecto: Baseline, Linear Regression, Random Forest, Gradient Boost e finalmente o LightGBM;

- Os vários modelos tiveram resultados dentro do esperado, exceptuando o Gradient Boost (48.33), que só se equipara a Baseline (87.21). Foi requisitado que todos tivessem como métrica REQM não superior a 48: LR = 47.17, RFR = 45.65, GB = 48.33 e LGBM = 47.37. Neste caso o Gradient Boost falhou em atender ao critério de sucesso estipulado;

- O melhor modelo de entre todos eles, foi o RFR (RandomForestRegressor), com 45.65. O que significa? O erro quadrático médio penaliza erros maiores, tendo como erro 45.65 significa que os pedidos oscilam entre 45.65 a mais ou a menos a cada hora;

- O modelo que recomendo para a empresa Sweet Lift Taxi, é o RandomForestRegressor por ter a métrica REQM mais baixa. Tem um impacto directo no negócio facilitando a redução do tempo de espera por ajuda a prever onde e quando a demanda vai subir. Pode também ajudar na maximização da facturação, revertendo assim mais revenue para o negócio. Permite a calibração da tarifação de maneira dinâmica com maior precisão, equilibrando a oferta e demanda sem afastar a clientela.

- Foram determinantes neste projecto a sazonalidade temporal, a defasagem histórica e a média movel.  

# Revisão da checklist

- [x]  O Jupyter Notebook está aberto.

- [x]  O código está livre de erros
- [x]  As células com o código foram organizadas em ordem de execução.
- [x]  Os dados foram baixados e preparados

- [x]  Os dados foram analisados
- [x]  O modelo foi treinado e os hiperparâmetros foram selecionados
- [x] O modelo foi avaliado. Uma conclusão foi fornecida
- [x] O REQM para o conjunto de teste não é maior que 48

In [ ]:
pip freeze > requirements.txt